# 04 · Node-perturbation comparison: `Cellina` vs `Celcomen` (LOO Myeloid, `crc_232`)

Benchmarks **Celcomen** ([Megas et al., 2025](https://github.com/Teichlab/celcomen)) against
**Cellina** on the **node-perturbation** task from the
[Cellina tutorial](../../../cellina/docs/tutorial.ipynb), on the **same slide** (`crc_232`)
under a **matched** leave-one-cell-type-out protocol.

**Node perturbation**: keep the spatial graph topology fixed, edit the *neighbours'* features —
shift the neighbourhood expression of a focal cell by the observed REF→CRC logFC, then ask the
model to predict the focal cell's response, scored against the observed cancer cells.

## Matched protocol (both models, same data, same cells)

| | Cellina | Celcomen |
|---|---|---|
| **Train on** | full slide minus `(Myeloid, CRC)` (`external_indexing`) | full slide minus `(Myeloid, CRC)` (held-out cells dropped as graph nodes) |
| **Perturbed at** | a shared **REF subpatch** of focal Myeloid | the **same** REF subpatch |
| **Predicts** | the shared focal REF-Myeloid cells | the **same** focal cells |
| **Scored vs** | observed CRC Myeloid | the **same** observed CRC Myeloid |

Both see the entire CRC microenvironment *except* `(Myeloid, CRC)`, so both are equally
in-domain — the only out-of-distribution element is the held-out Myeloid-in-CRC target, which
is what both must predict. Celcomen's Simcomen relaxation runs only on the subpatch (tractable,
and it avoids the high-dimensional spherical-embedding instability — see Section B).

## Cross-env structure

Celcomen lives in its own `environments/celcomen.yml` env (PyTorch-Geometric; no `scvi-tools`).
Run **Section A** in the `cellina` kernel, switch to `celcomen` for **Section B**, then run
**Section C** in either. Handoff is via cached `.h5ad` in `preds_celcomen/`.

## ⚠️ Two caveats this comparison rests on

1. **Celcomen has no cell-type labels.** It learns a single *global* gene–gene interaction
   matrix, so "leave-one-cell-type-out" can only mean *dropping those cells as graph nodes*
   (done here, to prevent the held-out CRC Myeloids from leaking into `G2G`). Because they are
   a small fraction of ~112k cells, the holdout barely perturbs what Celcomen learns — the LOO
   notion is inherently **weaker** for Celcomen than for Cellina (which conditions its latent on
   cell type). The holdout is applied identically to both regardless.
2. **Celcomen has no graph-level perturbation.** Simcomen imposes a perturbation as an
   *initial condition* and relaxes the whole field jointly to an energy minimum — nothing is
   clamped in the released code. To mimic Cellina's node perturbation we **clamp** the perturbed
   neighbours every step and relax only the focal cells, then back-map Celcomen's signed
   spherical embedding to a count-like quantity. These adaptations are why the comparison is
   informative but **not** like-for-like: Celcomen targets *in-silico gene-level* perturbations,
   not perturbations on the tissue graph.

## A. Cellina node perturbation  *(run in the `cellina` env)*

Full-slide training with `(Myeloid, CRC)` held out (the tutorial setup), then a node
perturbation evaluated on a **shared REF subpatch** of focal Myeloid cells.

In [1]:
import os
import sys

import numpy as np
import pandas as pd
import scanpy as sc
import torch
from scipy.stats import pearsonr

sys.path.append(os.path.abspath("."))                # local, self-contained scoring helpers
from utils import set_seed
from eval import make_comparison_adata, run_loo_eval, COUNTS_PER_K

from cellina import Cellina, make_neighbor_perturbation
from cellina._spatial_utils import spatial_neighbors, compute_spatial_features

set_seed(0)

CANDIDATE_PATHS = [
    "../../data/crc_wt_cosmx/crc_232.h5ad",
    os.path.join(os.environ.get("DATA_ROOT", "."), "datasets/crc/crc_232.h5ad"),
]
ADATA_PATH = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), CANDIDATE_PATHS[0])
ZENODO_URL = "https://zenodo.org/records/15574384/files/232.h5ad?download=1"

HOLDOUT_CT      = "Myeloid"
LABELS_KEY      = "coarse_type"
DOMAINS_KEY     = "typ"            # raw domain string ('232_REF' / '232_CRC' / ...), as in the tutorial
CONTROL_REGEX   = "REF"
TARGET_REGEX    = "CRC"
CONTROL_DOMAIN  = "232_REF"        # pseudobulk keys for the perturbation logFC
TARGET_DOMAIN   = "232_CRC"
N_PERT_GENES    = 200
SUBPATCH_N      = 300             # number of focal REF-Myeloid cells in the shared subpatch
                                  # (kept modest so Simcomen relaxation + calc_sphex stay stable)

PRED_DIR = "preds_celcomen"
os.makedirs(PRED_DIR, exist_ok=True)
print("data:", ADATA_PATH)

ModuleNotFoundError: No module named 'cellina'

In [ ]:
# --- Load + CRC preprocessing (tutorial-style) -----------------------------
adata = sc.read(ADATA_PATH, backup_url=ZENODO_URL)
adata.obs_names_make_unique()

label_to_coarse = {
    "epi1": "Epithelial", "epi2": "Epithelial", "epi3": "Epithelial", "epi4": "Epithelial",
    "fib1": "Fibroblast", "fib2": "Fibroblast",
    "EC": "Endothelial", "SMC": "Smooth_muscle",
    "BC": "B_cell", "PC_IgA": "Plasma_cell", "PC_IgG": "Plasma_cell", "PC_IgM": "Plasma_cell",
    "TC": "T_cell", "mye1": "Myeloid", "mye2": "Myeloid", "mast": "Mast_cell",
}
adata.obs["coarse_type"] = adata.obs["ist"].map(label_to_coarse)

adata = adata[~adata.obs[DOMAINS_KEY].isna()]
adata = adata[~adata.obs[LABELS_KEY].isna()]
sc.pp.filter_cells(adata, min_counts=3)
sc.pp.filter_genes(adata, min_counts=3)

adata.layers["counts"] = adata.X.copy()
sc.pp.highly_variable_genes(adata, layer="counts", flavor="seurat_v3", n_top_genes=2000, subset=True)
adata

/data/ddimitrov/software/miniforge3/envs/cellina_edge/lib/python3.10/site-packages/scanpy/preprocessing/_simple.py:165: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["n_counts"] = number


AnnData object with n_obs × n_vars = 111989 × 2000
    obs: 'fov', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'Width', 'Height', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD68_CK8_18', 'Max.CD68_CK8_18', 'Mean.CD298_B2M', 'Max.CD298_B2M', 'Mean.CD45', 'Max.CD45', 'Mean.DAPI', 'Max.DAPI', 'cell_id', 'Dash', 'ISH.concentration', 'Panel', 'Run_Tissue_name', 'Run_name', 'assay_type', 'dualfiles', 'tissue', 'version', 'slide_ID', 'CenterX_global_px', 'CenterY_global_px', 'cell_ID', 'unassignedTranscripts', 'median_RNA', 'RNA_quantile_0.75', 'RNA_quantile_0.8', 'RNA_quantile_0.85', 'RNA_quantile_0.9', 'RNA_quantile_0.95', 'RNA_quantile_0.99', 'nCount_RNA', 'nFeature_RNA', 'median_negprobes', 'negprobes_quantile_0.75', 'negprobes_quantile_0.8', 'negprobes_quantile_0.85', 'negprobes_quantile_0.9', 'negprobes_quantile_0.95', 'negprobes_quantile_0.99', 'nCount_negprobes', 'nFeature_negprobes', 'median_falsecode', 'falsecode_quantile_0.75', 'falsecode_quantile_0.8', 'falsecode_quantile

In [ ]:
# --- LOO split: hold out Myeloid cells in the CRC region (both models use this) -
is_tumor_region = adata.obs[DOMAINS_KEY].str.contains(TARGET_REGEX, regex=True)
is_holdout_ct   = adata.obs[LABELS_KEY] == HOLDOUT_CT

test_mask    = (is_tumor_region & is_holdout_ct)
test_idx     = np.where(test_mask.values)[0]
trainval_idx = np.setdiff1d(np.arange(adata.n_obs), test_idx)

adata.obs["is_holdout"] = False
adata.obs.iloc[test_idx, adata.obs.columns.get_loc("is_holdout")] = True

rng = np.random.default_rng(0)
n_val = max(1, int(0.1 * trainval_idx.size))
val_idx   = trainval_idx[rng.choice(trainval_idx.size, size=n_val, replace=False)]
train_idx = np.setdiff1d(trainval_idx, val_idx)
print(f"train={train_idx.size}  val={val_idx.size}  test(held-out Myeloid-CRC)={test_idx.size}")

train=96259  val=10695  test(held-out Myeloid-CRC)=5035


In [2]:
# --- Spatial neighbour graph + aggregated features (held-out cells masked) ---
sc.pp.normalize_total(adata, target_sum=COUNTS_PER_K)
sc.pp.log1p(adata)

adata.obsm["spatial"] = adata.obs[["CenterX_global_px", "CenterY_global_px"]].values
adata.obsp["spatial_connectivities_orig"] = spatial_neighbors(
    adata, bandwidth=100 / 0.12028, max_neighbours=200, standardize=False, inplace=False
)
spatial_neighbors(adata, bandwidth=100 / 0.12028, max_neighbours=200,
                  standardize=False, test_indices=test_idx)
compute_spatial_features(adata)

adata.X = adata.layers["counts"].copy()   # cellina expects raw counts in .X

NameError: name 'adata' is not defined

In [3]:
# --- Train Cellina on the full slide minus (Myeloid, CRC) -------------------
Cellina.setup_anndata(
    adata, batch_key=None, labels_key=LABELS_KEY, domains_key=DOMAINS_KEY,
    spatial_obsm_key="spatial_x", layer="counts",
)
cellina_args = dict(n_latent=64, use_observed_lib_size=True, condition_on_intrinsic=False,
                    classifier_lambda=1.0, discriminator_lambda=1.0, gene_likelihood="nb", n_layers=2)
model = Cellina(adata, **cellina_args)
model.train(
    max_epochs=100, batch_size=2048, check_val_every_n_epoch=1,
    early_stopping=True, early_stopping_patience=10,
    early_stopping_monitor="vae_loss_validation", devices=[0],
    datasplitter_kwargs={"external_indexing": [train_idx, val_idx, test_idx]},
    plan_kwargs={"lr": 1e-3, "normalize_losses": True},
)

NameError: name 'Cellina' is not defined

In [6]:
# --- Define the shared REF subpatch of focal Myeloid cells ------------------
# A contiguous spatial patch: the SUBPATCH_N REF-Myeloid cells closest to the REF-Myeloid
# centroid. Defined ONCE here and reused (as `is_focal`) by both models.
mask_ctrl_mye = (adata.obs[DOMAINS_KEY].str.contains(CONTROL_REGEX, regex=True) & is_holdout_ct).values
idx_ctrl_mye  = np.where(mask_ctrl_mye)[0]
coords = adata.obsm["spatial"]
centroid = coords[idx_ctrl_mye].mean(0)
order = np.argsort(((coords[idx_ctrl_mye] - centroid) ** 2).sum(1))
focal_idx = idx_ctrl_mye[order[:min(SUBPATCH_N, idx_ctrl_mye.size)]]

# Observed CRC-Myeloid target population (the held-out ground truth, same for both models).
mask_target = (is_tumor_region & is_holdout_ct).values
idx_target  = np.where(mask_target)[0]
print(f"focal REF-Myeloid (subpatch)={focal_idx.size}  |  observed CRC-Myeloid target={idx_target.size}")

focal REF-Myeloid (subpatch)=300  |  observed CRC-Myeloid target=5035


In [7]:
# --- Per-cell-type REF->CRC perturbation logFC (pseudobulk) -----------------
import scipy.sparse as sp
import decoupler as dc

def get_perturbation_logfc(adata, control_domain, holdout_domain, labels_key, domains_key):
    pdata = dc.pp.pseudobulk(adata=adata, sample_col=domains_key, groups_col=labels_key,
                             mode="sum", layer="counts")
    sc.pp.normalize_total(pdata, target_sum=1e4); sc.pp.log1p(pdata)
    cts = [ct for ct in pdata.obs[labels_key].unique()
           if ((pdata.obs[domains_key] == control_domain) & (pdata.obs[labels_key] == ct)).any()
           and ((pdata.obs[domains_key] == holdout_domain) & (pdata.obs[labels_key] == ct)).any()]
    rows = []
    for ct in cts:
        crc = pdata[(pdata.obs[domains_key] == holdout_domain) & (pdata.obs[labels_key] == ct)].X
        ref = pdata[(pdata.obs[domains_key] == control_domain) & (pdata.obs[labels_key] == ct)].X
        crc_m = np.asarray(crc.mean(0)).ravel() if sp.issparse(crc) else crc.mean(0).ravel()
        ref_m = np.asarray(ref.mean(0)).ravel() if sp.issparse(ref) else ref.mean(0).ravel()
        rows.append(pd.Series(crc_m - ref_m, index=pdata.var_names, name=ct))
    return pd.concat(rows, axis=1).T

def get_global_perturbation_logfc(adata, control_domain, holdout_domain, labels_key, domains_key, holdout_ct):
    sub = adata[adata.obs[labels_key] != holdout_ct]
    pdata = dc.pp.pseudobulk(adata=sub, sample_col=domains_key, groups_col=None, mode="sum", layer="counts")
    sc.pp.normalize_total(pdata, target_sum=1e4); sc.pp.log1p(pdata)
    h = pdata[pdata.obs[domains_key] == holdout_domain].X
    c = pdata[pdata.obs[domains_key] == control_domain].X
    h_m = np.asarray(h.mean(0)).ravel() if sp.issparse(h) else h.mean(0).ravel()
    c_m = np.asarray(c.mean(0)).ravel() if sp.issparse(c) else c.mean(0).ravel()
    return pd.Series(h_m - c_m, index=pdata.var_names)

domain_logfc_df     = get_perturbation_logfc(adata, CONTROL_DOMAIN, TARGET_DOMAIN, LABELS_KEY, DOMAINS_KEY)
global_logfc_series = get_global_perturbation_logfc(adata, CONTROL_DOMAIN, TARGET_DOMAIN, LABELS_KEY, DOMAINS_KEY, HOLDOUT_CT)
domain_logfc_df.loc[HOLDOUT_CT, global_logfc_series.index] = global_logfc_series  # holdout uses GLOBAL shift

logfc_series_dict = {ct: domain_logfc_df.loc[ct][domain_logfc_df.loc[ct].abs().nlargest(N_PERT_GENES).index]
                     for ct in domain_logfc_df.index}
print("perturbation cell types:", list(logfc_series_dict))

perturbation cell types: ['Endothelial', 'Epithelial', 'Fibroblast', 'Myeloid', 'Plasma_cell', 'Smooth_muscle', 'T_cell']


In [8]:
# --- Cellina node perturbation on the focal subpatch ------------------------
adata.X = adata.layers["counts"].copy()
sc.pp.normalize_total(adata, target_sum=COUNTS_PER_K)
sc.pp.log1p(adata)

make_neighbor_perturbation(
    adata, perturbations=logfc_series_dict, groupby=LABELS_KEY,
    obsm_key_out="spatial_x_cf", base=np.e, renormalize=True, add_shift=True,
)
cellina_cf = model.get_perturbed_expression(
    adata=adata, indices=focal_idx, spatial_obsm_key="spatial_x_cf",
    batch_size=2048, library_size=COUNTS_PER_K,
)
cellina_ctrl = model.get_normalized_expression(adata[focal_idx], library_size=COUNTS_PER_K, batch_size=2048)

INFO     Received view of anndata, making copy.                                                                    
INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


/data/ddimitrov/software/miniforge3/envs/cellina_edge/lib/python3.10/site-packages/scvi/data/fields/_layer_field.py:115: UserWarning: Training will be faster when sparse matrix is formatted as CSR. It is safe to cast before model initialization.
  _verify_and_correct_data_format(adata, self.attr_name, self.attr_key)


In [9]:
# --- Cache observed + cellina predictions (shared focal cells) --------------
observed = make_comparison_adata(
    adata.layers["counts"][focal_idx, :],          # control arm: focal REF Myeloid
    adata.layers["counts"][idx_target, :],         # target arm: observed CRC Myeloid (held out)
    adata.var_names, control_label="REF", perturbed_label="CRC",
)
observed.obs[LABELS_KEY] = HOLDOUT_CT
observed.write(os.path.join(PRED_DIR, "observed.h5ad"))

cellina_pred = make_comparison_adata(
    cellina_ctrl, cellina_cf, adata.var_names, control_label="REF", perturbed_label="CRC",
)
cellina_pred.obs[LABELS_KEY] = HOLDOUT_CT          # both arms are the focal REF Myeloid
cellina_pred.write(os.path.join(PRED_DIR, "cellina_pred.h5ad"))
print("cached observed + cellina_pred", observed.shape, cellina_pred.shape)

... storing 'coarse_type' as categorical
... storing 'coarse_type' as categorical


cached observed + cellina_pred (5335, 2000) (600, 2000)


In [10]:
# --- Cache full-slide Celcomen inputs (Section B trains G2G on the full slide) -
# Celcomen learns its global interaction matrices on the full slide MINUS the held-out
# (Myeloid, CRC) nodes (`is_holdout`), then relaxes only the focal subpatch (`is_focal`).
cc = adata.copy()
cc.X = cc.layers["counts"].copy()
cc.obs["is_focal"] = False
cc.obs.iloc[focal_idx, cc.obs.columns.get_loc("is_focal")] = True   # shared focal subpatch
for k in ("spatial_x", "spatial_x_cf"):
    cc.obsm.pop(k, None)
keep_obs = ["coarse_type", DOMAINS_KEY, "is_holdout", "is_focal"]
cc.obs = cc.obs[keep_obs]
logfc_full = pd.DataFrame(0.0, index=list(adata.var_names), columns=list(domain_logfc_df.index))
for ct, s in logfc_series_dict.items():
    logfc_full.loc[s.index, ct] = s.values
cc.varm["pert_logfc"] = logfc_full.values
cc.uns["pert_logfc_celltypes"] = list(logfc_full.columns)
cc.uns["holdout_ct"] = HOLDOUT_CT
cc.uns["domains_key"] = DOMAINS_KEY
if "counts" in cc.layers:
    del cc.layers["counts"]            # cc.X already holds raw counts
cc.obsp.clear()                        # drop the (large) full-slide adjacency; B rebuilds kNN
cc.write(os.path.join(PRED_DIR, "celcomen_inputs.h5ad"))
print("cached celcomen_inputs (full slide)", cc.shape)

... storing 'coarse_type' as categorical


cached celcomen_inputs (full slide) (111989, 2000)


In [11]:
# --- Cellina node-perturbation result on the focal subpatch (sanity) --------
def _norm_counts(x, eps=1e-8, scale=COUNTS_PER_K):
    x = np.asarray(x); return x / (x.sum(1, keepdims=True) + eps) * scale
def _safe_log2fc(a, b, eps=1e-6): return np.log2((np.asarray(a) + eps) / (np.asarray(b) + eps))

ctrl = _norm_counts(observed[observed.obs["typ_clean"] == "REF"].X)
tgt  = _norm_counts(observed[observed.obs["typ_clean"] == "CRC"].X)
cf   = _norm_counts(cellina_pred[cellina_pred.obs["typ_clean"] == "CRC"].X)
true_lfc = _safe_log2fc(tgt.mean(0), ctrl.mean(0))
pred_lfc = _safe_log2fc(cf.mean(0),  ctrl.mean(0))
deg = np.argsort(-np.abs(true_lfc))[:20]
r, _ = pearsonr(true_lfc[deg], pred_lfc[deg])
print(f"Cellina node-pert (focal Myeloid subpatch): Pearson r (top-20) = {r:.3f}")

Cellina node-pert (focal Myeloid subpatch): Pearson r (top-20) = 0.642


## B. Celcomen node-perturbation adapter  *(run in the `celcomen` env)*

> Switch the notebook kernel to **`celcomen`** before running this section.

1. **Train Celcomen on the full slide minus `(Myeloid, CRC)`** (held-out cells dropped as graph
   nodes) → learns the global interaction matrices `G2G` / `G2G_intra`. Same data Cellina saw.
2. Transfer the frozen weights to **Simcomen**, restricted to the **shared focal subpatch**
   (`is_focal` + their neighbours).
3. Shift the subpatch **neighbours'** expression by the same per-cell-type REF→CRC logFC Cellina
   used (Myeloid → global), **clamp** them, and relax only the focal cells.
4. Back-map the relaxed focal cells from Celcomen's signed *spherical* embedding to a count-like
   quantity (× stored L2 norm, `expm1`, clip ≥ 0) — an approximation introduced by the adapter.

> **Numerical note.** `calc_sphex` chains divisions by `sin` of preceding angles and is unstable
> at high gene counts (the Celcomen tutorial used ~450 genes; here we carry 2000 HVGs). Keeping
> the relaxation on the small focal subpatch helps; if it still NaNs/diverges, reduce the gene
> set for the Celcomen side (e.g. union of perturbed genes + top HVGs of the subpatch) and re-run.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import scanpy as sc
import torch
from tqdm import tqdm
from sklearn.neighbors import kneighbors_graph

from celcomen.models.celcomen import celcomen
from celcomen.models.simcomen import simcomen
from celcomen.utils.helpers import normalize_g2g, calc_sphex

PRED_DIR = "preds_celcomen"
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
K_NN     = 6          # Celcomen's sparse kNN spatial graph (tutorial default)
ZMFT     = 1e-1
SEED     = 0
np.random.seed(SEED); torch.manual_seed(SEED)

cc = sc.read(os.path.join(PRED_DIR, "celcomen_inputs.h5ad"))
HOLDOUT_CT = cc.uns["holdout_ct"]
n_genes    = cc.n_vars
is_holdout = cc.obs["is_holdout"].values.astype(bool)    # (Myeloid, CRC) — excluded from G2G
is_focal   = cc.obs["is_focal"].values.astype(bool)      # shared focal REF Myeloid
print(f"full slide={cc.n_obs} x {n_genes} | held-out nodes={is_holdout.sum()} | "
      f"focal={is_focal.sum()} | device={DEVICE}")

/data/ddimitrov/software/miniforge3/envs/celcomen/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


full slide=111989 x 2000 | held-out nodes=5035 | focal=300 | device=cuda


In [2]:
# --- Expression on the unit hypersphere (log-normalised, then per-cell L2) ---
def to_lognorm(counts):
    a = sc.AnnData(np.asarray(counts.todense() if hasattr(counts, "todense") else counts, dtype=np.float32))
    sc.pp.normalize_total(a, target_sum=1e4); sc.pp.log1p(a)
    return a.X

lognorm = to_lognorm(cc.X)                               # (cells, genes)
norm_factor = np.sqrt((lognorm ** 2).sum(1, keepdims=True)); norm_factor[norm_factor == 0] = 1.0
base_expr = (lognorm / norm_factor).astype("float32")    # unit-L2 rows

In [3]:
# --- Train Celcomen G2G on the full slide MINUS the held-out nodes ----------
# Dropping (Myeloid, CRC) as nodes is the leakage-free analog of Cellina masking test_indices
# from its adjacency: those cells never inform the learned interactions, nor act as neighbours.
LR_CC, EPOCHS_CC = 1e-1, 200
keep = ~is_holdout
coords_keep = cc.obsm["spatial"][keep]
expr_keep   = torch.from_numpy(base_expr[keep]).to(DEVICE)
adj = kneighbors_graph(coords_keep, K_NN, include_self=False).toarray()
edge_keep = torch.from_numpy(np.array(np.where(adj))).long().to(DEVICE)

g0 = np.random.uniform(size=(n_genes, n_genes)).astype("float32"); g0 = normalize_g2g((g0 + g0.T) / 2)
cc_model = celcomen(input_dim=n_genes, output_dim=n_genes, n_neighbors=K_NN, seed=SEED)
cc_model.set_g2g(torch.from_numpy(g0)); cc_model.set_g2g_intra(torch.from_numpy(g0.copy()))
cc_model.to(DEVICE)

opt = torch.optim.SGD(cc_model.parameters(), lr=LR_CC, momentum=0)
cc_losses = []
for epoch in tqdm(range(EPOCHS_CC)):
    cc_model.set_gex(expr_keep)
    msg, msg_intra, log_z = cc_model(edge_keep, 1)
    loss = -(-log_z + ZMFT * torch.trace(msg @ cc_model.gex.T) + ZMFT * torch.trace(msg_intra @ cc_model.gex.T))
    loss.backward(); opt.step(); opt.zero_grad()
    cc_model.conv1.lin.weight = torch.nn.Parameter(normalize_g2g(cc_model.conv1.lin.weight), requires_grad=True)
    cc_model.lin.weight       = torch.nn.Parameter(normalize_g2g(cc_model.lin.weight), requires_grad=True)
    opt = torch.optim.SGD(cc_model.parameters(), lr=LR_CC, momentum=0)
    cc_losses.append(float(loss.detach().cpu().numpy().ravel()[0]))
print(f"Celcomen G2G trained (full slide - holdout): loss {cc_losses[0]:.1f} -> {cc_losses[-1]:.1f}")

  0%|          | 0/200 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 42.62 GiB. GPU 0 has a total capacity of 23.65 GiB of which 16.75 GiB is free. Including non-PyTorch memory, this process has 6.89 GiB memory in use. Of the allocated memory 2.44 GiB is allocated by PyTorch, and 4.01 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# --- Extract the focal subpatch for Simcomen (focal + their kNN neighbours) --
# Restricted to non-held-out cells, so the held-out Myeloids never enter the subpatch either.
keep_idx   = np.where(keep)[0]
focal_keep = is_focal[keep]                              # focal mask within the kept set
nbr = kneighbors_graph(coords_keep, K_NN, include_self=False)
nbr_of_focal = np.unique(nbr[focal_keep].nonzero()[1])
region_local = np.union1d(np.where(focal_keep)[0], nbr_of_focal)   # indices into the kept set

reg_expr_base = base_expr[keep_idx[region_local]]
reg_lognorm   = lognorm[keep_idx[region_local]]
reg_nf        = norm_factor[keep_idx[region_local]]
reg_labels    = cc.obs["coarse_type"].astype(str).values[keep_idx[region_local]]
reg_is_focal  = is_focal[keep_idx[region_local]]
reg_coords    = coords_keep[region_local]
adj_r = kneighbors_graph(reg_coords, K_NN, include_self=False).toarray()
edge_r = torch.from_numpy(np.array(np.where(adj_r))).long().to(DEVICE)
print(f"subpatch region={region_local.size} cells | focal within={reg_is_focal.sum()}")

In [ ]:
# --- Perturb the subpatch NEIGHBOURS to CRC-like; clamp; relax focal cells ---
logfc = pd.DataFrame(cc.varm["pert_logfc"], index=list(cc.var_names),
                     columns=list(cc.uns["pert_logfc_celltypes"]))
shift = np.zeros_like(reg_lognorm)
for ct in logfc.columns:
    rows = (reg_labels == ct) & (~reg_is_focal)          # perturb neighbours only
    if rows.any():
        shift[rows] = logfc[ct].values[None, :]
pert_lognorm = np.clip(reg_lognorm + shift, 0, None)
pert_nf = np.sqrt((pert_lognorm ** 2).sum(1, keepdims=True)); pert_nf[pert_nf == 0] = 1.0
pert_base = pert_lognorm / pert_nf

init_expr = reg_expr_base.copy()
init_expr[~reg_is_focal] = pert_base[~reg_is_focal].astype("float32")
init_sphex  = calc_sphex(torch.from_numpy(init_expr.astype("float32")))
clamp       = torch.from_numpy((~reg_is_focal)).to(DEVICE)
sphex_fixed = init_sphex.clone().to(DEVICE)

LR_SIM, EPOCHS_SIM = 1e-3, 50
sim = simcomen(input_dim=n_genes, output_dim=n_genes, n_neighbors=K_NN, seed=SEED)
sim.set_g2g(cc_model.conv1.lin.weight.clone().detach())
sim.set_g2g_intra(cc_model.lin.weight.clone().detach())
sim.set_sphex(init_sphex.clone()); sim.to(DEVICE)

opt = torch.optim.SGD(sim.parameters(), lr=LR_SIM, momentum=0)
sim_losses, orig_gex = [], None
for epoch in tqdm(range(EPOCHS_SIM)):
    msg, msg_intra, log_z = sim(edge_r, 1)
    if epoch == 0:
        orig_gex = sim.gex.clone().detach().cpu().numpy()
    loss = -(-log_z + ZMFT * torch.trace(msg @ sim.gex.T) + ZMFT * torch.trace(msg_intra @ sim.gex.T))
    loss.backward(); opt.step(); opt.zero_grad()
    with torch.no_grad():                                # re-clamp the perturbed neighbours
        sim.sphex[clamp] = sphex_fixed[clamp]
    sim_losses.append(float(loss.detach().cpu().numpy().ravel()[0]))
with torch.no_grad():
    pred_gex = sim.calc_gex(sim.sphex).detach().cpu().numpy()
print(f"Simcomen relaxed (focal only): loss {sim_losses[0]:.1f} -> {sim_losses[-1]:.1f}")

In [ ]:
# --- Back-map spherical embedding -> count-like, build celcomen_pred --------
sys.path.append(os.path.abspath("."))
from eval import make_comparison_adata

def to_counts(gex_rows, nf_rows):
    return np.clip(np.expm1(gex_rows * nf_rows), 0, None)   # undo L2 norm, then undo log1p

ctrl_counts = to_counts(orig_gex[reg_is_focal], reg_nf[reg_is_focal])   # focal REF reconstruction
pert_counts = to_counts(pred_gex[reg_is_focal], reg_nf[reg_is_focal])   # focal relaxed prediction

celcomen_pred = make_comparison_adata(
    ctrl_counts, pert_counts, cc.var_names, control_label="REF", perturbed_label="CRC",
)
celcomen_pred.obs["coarse_type"] = HOLDOUT_CT
celcomen_pred.write(os.path.join(PRED_DIR, "celcomen_pred.h5ad"))
print("cached celcomen_pred", celcomen_pred.shape)

## C. Score — matched comparison  *(either env)*

Both models predict the **same** focal REF-Myeloid subpatch and are scored against the **same**
observed CRC-Myeloid population, with `run_loo_eval` (top-50 observed-effect genes,
`predicted_logfc_baseline="observed_control"` — both ground-truth and predicted logFC measured
against the same observed REF population, as in the Cellina tutorial).

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import scanpy as sc

sys.path.append(os.path.abspath("."))
from eval import run_loo_eval
from metrics import get_perturbation_effects

PRED_DIR, PERT_COL, CONTROL_LABEL = "preds_celcomen", "typ_clean", "REF"
observed      = sc.read(os.path.join(PRED_DIR, "observed.h5ad"))
cellina_pred  = sc.read(os.path.join(PRED_DIR, "cellina_pred.h5ad"))
celcomen_pred = sc.read(os.path.join(PRED_DIR, "celcomen_pred.h5ad"))

comparison = pd.DataFrame([
    run_loo_eval(observed, cellina_pred,  pert_col=PERT_COL, control_label=CONTROL_LABEL).iloc[0].rename("cellina"),
    run_loo_eval(observed, celcomen_pred, pert_col=PERT_COL, control_label=CONTROL_LABEL).iloc[0].rename("celcomen"),
])
comparison

In [ ]:
# --- Observed vs predicted logFC scatter (both models, shared focal cells) --
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 5.2), sharex=True, sharey=True)
for ax, (name, pred) in zip(axes, [("cellina", cellina_pred), ("celcomen", celcomen_pred)]):
    o_eff, p_eff = get_perturbation_effects(
        observed, pred, pert_col=PERT_COL, control_label=CONTROL_LABEL,
        mode="logfc", predicted_logfc_baseline="observed_control",
    )
    o, p = o_eff.loc["CRC"].values, p_eff.loc["CRC"].values
    deg = np.argsort(-np.abs(o))[:50]
    ax.scatter(o, p, s=8, color="lightgray", alpha=0.4)
    ax.scatter(o[deg], p[deg], s=28, color="#d62728")
    lims = [min(o.min(), p.min()), max(o.max(), p.max())]
    ax.plot(lims, lims, "k--", lw=1, alpha=0.6)
    ax.set_title(f"{name} (focal Myeloid subpatch)"); ax.set_xlabel("Observed logFC (REF->CRC)")
axes[0].set_ylabel("Predicted logFC")
plt.tight_layout(); plt.show()

## Notes / interpretation

- This is now a **matched** comparison: identical training data (full slide minus
  `(Myeloid, CRC)`), the same shared focal cells, and the same observed CRC-Myeloid ground truth.
- Read Celcomen's numbers with the two caveats above in mind: (1) its global interaction matrix
  makes the cell-type holdout weaker than Cellina's, and (2) its prediction comes from a clamped
  joint energy relaxation on a sparse kNN graph + a spherical→count back-mapping, not a
  conditional decode under a held-fixed neighbourhood. The comparison is informative but **not**
  a like-for-like graph-perturbation baseline — Celcomen targets *in-silico gene-level*
  perturbations, not perturbations on the tissue graph.
- Knobs if you iterate: `SUBPATCH_N`, `K_NN`, `EPOCHS_SIM`/`LR_SIM`, `EPOCHS_CC`, `N_PERT_GENES`.